# 2025 cost-center address key
Reads sheets `1`–`8` from `controllable_costs.xlsx`. The screenshot shows cost centers under **Contract Name** and facilities under **Address**. The output is a separate `2025_costcenter_key.xlsx` with `Key` and `Review` tabs; the source workbook is not changed.


In [ ]:
from pathlib import Path
import pandas as pd

# Set this to the workbook's location if it is not beside the notebook.
source_path = Path('controllable_costs.xlsx')
output_path = source_path.with_name('2025_costcenter_key.xlsx')
sheet_names = [str(i) for i in range(1, 9)]

assert source_path.is_file(), f'File not found: {source_path.resolve()}'
assert source_path.resolve() != output_path.resolve()
print('Reading:', source_path.resolve())


In [ ]:
def tidy(value):
    return ' '.join(str(value).split()).strip()

book = pd.ExcelFile(source_path)
missing = sorted(set(sheet_names) - set(book.sheet_names))
if missing:
    raise ValueError(f'Missing numbered sheets: {missing}')

parts = []
for name in sheet_names:
    preview = pd.read_excel(book, sheet_name=name, header=None, nrows=50,
                            dtype=str, keep_default_na=False)
    header_row = next((i for i, row in preview.iterrows()
                       if {'contract name', 'address'} <= {tidy(v).casefold() for v in row}), None)
    if header_row is None:
        raise ValueError(f'Sheet {name}: Contract Name and Address headers not found in first 50 rows')
    headers = [tidy(v).casefold() for v in preview.iloc[header_row]]
    cc_col = headers.index('contract name')
    address_col = headers.index('address')
    data = pd.read_excel(book, sheet_name=name, header=header_row,
                         usecols=[cc_col, address_col], dtype=str, keep_default_na=False)
    columns = {tidy(col).casefold(): col for col in data.columns}
    part = pd.DataFrame({
        'Cost Center': data[columns['contract name']].map(tidy).str.upper(),
        'Address': data[columns['address']].map(tidy),
        'Source Sheet': name,
        'Source Row': [f'{name}!{header_row + i + 2}' for i in range(len(data))],
    })
    valid_code = part['Cost Center'].str.fullmatch(r'[A-Z0-9]{6,20}') & part['Cost Center'].str.contains(r'\d')
    parts.append(part.loc[valid_code].copy())
    print(f'Sheet {name}: {valid_code.sum():,} rows with a cost-center-like code')

rows = pd.concat(parts, ignore_index=True)
rows['Address Match'] = rows['Address'].str.casefold()
print(f'Total source rows with cost-center-like codes: {len(rows):,}')


In [ ]:
# A code with multiple distinct nonblank addresses needs a human choice.
with_address = rows.loc[rows['Address'].ne('')].copy()
address_count = with_address.groupby('Cost Center')['Address Match'].nunique()
conflicting_codes = set(address_count.loc[address_count.gt(1)].index)

def example_rows(series):
    return ', '.join(series.head(5))

key = (with_address.loc[~with_address['Cost Center'].isin(conflicting_codes)]
       .groupby('Cost Center', as_index=False)
       .agg(Address=('Address', 'first'),
            Source_Sheets=('Source Sheet', lambda s: ', '.join(sorted(set(s)))),
            Source_Rows=('Source Row', 'size'),
            Example_Rows=('Source Row', example_rows))
       .rename(columns={'Source_Sheets': 'Source sheets', 'Source_Rows': 'Source rows',
                        'Example_Rows': 'Example source rows'})
       .sort_values('Cost Center'))

review_rows = rows.loc[rows['Address'].eq('') | rows['Cost Center'].isin(conflicting_codes)].copy()
review_rows['Issue'] = review_rows['Cost Center'].map(
    lambda code: 'Conflicting addresses' if code in conflicting_codes else 'Blank address')
review_rows.loc[review_rows['Address'].eq(''), 'Issue'] = 'Blank address'
review = (review_rows.groupby(['Issue', 'Cost Center', 'Address'], as_index=False)
          .agg(Source_rows=('Source Row', 'size'),
               Source_sheets=('Source Sheet', lambda s: ', '.join(sorted(set(s)))),
               Example_source_rows=('Source Row', example_rows))
          .rename(columns={'Source_rows': 'Source rows', 'Source_sheets': 'Source sheets',
                           'Example_source_rows': 'Example source rows'})
          .sort_values(['Cost Center', 'Issue', 'Address']))

print(f'Key: {len(key):,} unambiguous cost centers')
print(f'Review: {len(review):,} address variants or blanks; {len(conflicting_codes):,} conflicting codes')
display(key.head(10))
display(review.head(10))


In [ ]:
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    key.to_excel(writer, sheet_name='Key', index=False)
    review.to_excel(writer, sheet_name='Review', index=False)
    for sheet in writer.sheets.values():
        sheet.freeze_panes = 'A2'
        sheet.auto_filter.ref = sheet.dimensions
        for column, width in {'A': 25, 'B': 55, 'C': 22, 'D': 18, 'E': 35, 'F': 35}.items():
            sheet.column_dimensions[column].width = width

print('Saved:', output_path.resolve())
print('Check the Review tab before using the Key for 2025 facility mapping.')
